In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import BytesIO

BASE_URL = (
    "https://www.ncei.noaa.gov/pub/data/"
    "swdi/stormevents/csvfiles/"
)

response = requests.get(BASE_URL, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

files = [
    a.get("href")
    for a in soup.find_all("a", href=True)
    if a.get("href", "").startswith("StormEvents_details")
    and a.get("href", "").endswith(".csv.gz")
]

flood_types = [
    "Flood",
    "Flash Flood",
    "Coastal Flood",
    "Lakeshore Flood"
]

all_floods = []

for year in range(2000, 2027):

    matching = [
        f for f in files
        if f"_d{year}_" in f
    ]

    if not matching:
        print(f"No file found for {year}")
        continue

    filename = sorted(matching)[-1]

    print(f"Downloading {year}...")

    r = requests.get(BASE_URL + filename, timeout=180)
    r.raise_for_status()

    yearly_df = pd.read_csv(
        BytesIO(r.content),
        compression="gzip",
        low_memory=False
    )

    yearly_df = yearly_df[
        yearly_df["EVENT_TYPE"].isin(flood_types)
    ].copy()

    all_floods.append(yearly_df)

flood_df = pd.concat(
    all_floods,
    ignore_index=True
)

print("Total flood records:", len(flood_df))
print(flood_df["EVENT_TYPE"].value_counts())

Total flood records: 170420
EVENT_TYPE
Flash Flood        102345
Flood               63536
Coastal Flood        4180
Lakeshore Flood       359
Name: count, dtype: int64


In [2]:
import numpy as np

def parse_damage(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper().replace(",", "")

    if value in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    multipliers = {
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000
    }

    suffix = value[-1]

    if suffix in multipliers:
        number = value[:-1]

        if not number:
            return np.nan

        return pd.to_numeric(
            number, errors="coerce"
        ) * multipliers[suffix]

    return pd.to_numeric(value, errors="coerce")


# Convert damage columns
for col in ["DAMAGE_PROPERTY", "DAMAGE_CROPS"]:
    flood_df[col + "_NUM"] = (
        flood_df[col].apply(parse_damage)
    )

# Calculate total economic damage
flood_df["total_damage_usd"] = flood_df[
    ["DAMAGE_PROPERTY_NUM", "DAMAGE_CROPS_NUM"]
].sum(axis=1, min_count=1)

# Convert fatalities and injuries
casualty_columns = [
    "DEATHS_DIRECT",
    "DEATHS_INDIRECT",
    "INJURIES_DIRECT",
    "INJURIES_INDIRECT"
]

for col in casualty_columns:
    flood_df[col] = pd.to_numeric(
        flood_df[col],
        errors="coerce"
    ).fillna(0)

flood_df["total_deaths"] = (
    flood_df["DEATHS_DIRECT"]
    + flood_df["DEATHS_INDIRECT"]
)

flood_df["total_injuries"] = (
    flood_df["INJURIES_DIRECT"]
    + flood_df["INJURIES_INDIRECT"]
)

print(flood_df[
    [
        "EVENT_ID",
        "EVENT_TYPE",
        "total_damage_usd",
        "total_deaths",
        "total_injuries"
    ]
].head())

   EVENT_ID EVENT_TYPE  total_damage_usd  total_deaths  total_injuries
0   5128936      Flood               NaN             0               0
1   5128937      Flood               NaN             0               0
2   5129280      Flood               NaN             0               0
3   5129281      Flood               NaN             0               0
4   5129282      Flood               NaN             0               0


In [3]:
# Convert dates
flood_df["BEGIN_DATE_TIME"] = pd.to_datetime(
    flood_df["BEGIN_DATE_TIME"],
    errors="coerce"
)

flood_df["END_DATE_TIME"] = pd.to_datetime(
    flood_df["END_DATE_TIME"],
    errors="coerce"
)

# Aggregate flood reports by episode
flood_episodes = (
    flood_df.groupby("EPISODE_ID")
    .agg(
        start_date=("BEGIN_DATE_TIME", "min"),
        end_date=("END_DATE_TIME", "max"),
        total_damage_usd=("total_damage_usd", "sum"),
        total_deaths=("total_deaths", "sum"),
        total_injuries=("total_injuries", "sum"),
        affected_records=("EVENT_ID", "nunique"),
        states=(
            "STATE",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        ),
        flood_types=(
            "EVENT_TYPE",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        )
    )
    .reset_index()
)

# Filter for major flood episodes
major_floods = flood_episodes[
    (flood_episodes["total_damage_usd"] >= 100_000_000)
    | (flood_episodes["total_deaths"] >= 10)
    | (flood_episodes["total_injuries"] >= 50)
].copy()

major_floods = major_floods.sort_values(
    "total_damage_usd",
    ascending=False
)

print("Major flood episodes:", len(major_floods))

/var/folders/88/372_s7qj62b3qk7xqrcjxflw0000gn/T/ipykernel_21504/2566716804.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  flood_df["BEGIN_DATE_TIME"] = pd.to_datetime(
/var/folders/88/372_s7qj62b3qk7xqrcjxflw0000gn/T/ipykernel_21504/2566716804.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  flood_df["END_DATE_TIME"] = pd.to_datetime(


Major flood episodes: 158


In [4]:
# Rename episode identifier
major_floods = major_floods.rename(
    columns={"EPISODE_ID": "episode_id"}
)

# Create flood identifier
major_floods["flood_id"] = (
    "FLOOD_"
    + major_floods["episode_id"].astype(str)
)

# Calculate duration
major_floods["duration_days"] = (
    major_floods["end_date"]
    - major_floods["start_date"]
).dt.total_seconds() / 86400

# Select final columns
final_columns = [
    "flood_id",
    "episode_id",
    "start_date",
    "end_date",
    "states",
    "flood_types",
    "total_damage_usd",
    "total_deaths",
    "total_injuries",
    "affected_records",
    "duration_days"
]

major_floods = major_floods[final_columns]

print(major_floods.head(20))

            flood_id  episode_id          start_date            end_date  \
19462   FLOOD_119753      119753 2017-08-25 23:45:00 2017-08-31 12:00:00   
10752    FLOOD_68471       68471 2012-10-28 18:00:00 2012-10-31 17:00:00   
19610   FLOOD_120357      120357 2017-09-20 00:00:00 2017-09-22 08:00:00   
17709   FLOOD_109514      109514 2016-08-11 09:00:00 2016-08-27 16:00:00   
19459   FLOOD_119746      119746 2017-08-26 12:30:00 2017-08-31 23:59:00   
8641     FLOOD_53127       53127 2011-05-01 00:00:00 2011-05-31 23:59:00   
14220    FLOOD_89491       89491 2014-08-11 15:00:00 2014-08-13 00:00:00   
32335   FLOOD_196694      196694 2024-09-25 18:00:00 2024-09-29 06:00:00   
5857     FLOOD_37606       37606 2010-05-01 10:00:00 2010-05-04 18:00:00   
17810   FLOOD_110040      110040 2016-08-14 06:00:00 2016-08-31 23:59:00   
28793   FLOOD_173247      173247 2022-08-21 21:31:00 2022-08-22 21:00:00   
9214     FLOOD_56116       56116 2011-08-28 10:30:00 2011-08-31 04:48:00   
2982     FLO

In [5]:
major_floods.to_csv(
    "major_floods_2000_2026.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!
